In [ ]:
# STEP 1: Install Dependencies
import os

!rm -rf /content/IDM-VTON
!git clone --depth 1 https://github.com/yisol/IDM-VTON.git /content/IDM-VTON

!pip uninstall -y sentence-transformers
!pip install diffusers==0.25.0 accelerate==0.25.0 peft==0.10.0 transformers==4.36.2 bitsandbytes "huggingface_hub<0.24.0"
!pip install gradio==4.24.0 einops omegaconf fvcore cloudpickle pycocotools av scikit-image onnxruntime

print('Setup complete. Restart the Colab runtime once, then continue with the next cell.')
os.kill(os.getpid(), 9)


In [ ]:
# STEP 2: Download the FP16 models
%cd /content/IDM-VTON
import os
from huggingface_hub import hf_hub_download, snapshot_download

!rm -rf ckpt/humanparsing/*.onnx
!rm -rf ckpt/openpose/ckpts/*.pth
os.makedirs('ckpt/densepose', exist_ok=True)
os.makedirs('ckpt/humanparsing', exist_ok=True)
os.makedirs('ckpt/openpose/ckpts', exist_ok=True)
os.makedirs('ckpt/image_encoder', exist_ok=True)

!wget -q -nc -O ckpt/densepose/model_final_162be9.pkl https://dl.fbaipublicfiles.com/densepose/densepose_rcnn_R_50_FPN_s1x/165712039/model_final_162be9.pkl
hf_hub_download(repo_id='yisol/IDM-VTON', filename='humanparsing/parsing_atr.onnx', local_dir='ckpt', local_dir_use_symlinks=False)
hf_hub_download(repo_id='yisol/IDM-VTON', filename='humanparsing/parsing_lip.onnx', local_dir='ckpt', local_dir_use_symlinks=False)
hf_hub_download(repo_id='yisol/IDM-VTON', filename='openpose/ckpts/body_pose_model.pth', local_dir='ckpt', local_dir_use_symlinks=False)
snapshot_download(repo_id='camenduru/IDM-VTON-F16', local_dir='ckpt/IDM-VTON-F16', local_dir_use_symlinks=False)
print('Models downloaded successfully.')


In [ ]:
# STEP 3: Patch the official Gradio demo and launch it with a public URL
%cd /content/IDM-VTON
!git checkout gradio_demo/app.py
from pathlib import Path

app = Path('gradio_demo/app.py')
src = app.read_text()

patch_gradio = """
import gradio_client.utils
if not hasattr(gradio_client.utils, '_original_json_schema_to_python_type'):
    gradio_client.utils._original_json_schema_to_python_type = gradio_client.utils._json_schema_to_python_type
def safe_json_schema_to_python_type(schema, defs):
    if isinstance(schema, bool):
        return 'Any'
    return gradio_client.utils._original_json_schema_to_python_type(schema, defs)
gradio_client.utils._json_schema_to_python_type = safe_json_schema_to_python_type
"""
src = patch_gradio + '\n' + src

patch_device = """
TryonPipeline.device = property(lambda self: torch.device(device))
TryonPipeline._execution_device = property(lambda self: torch.device(device))
"""
src = src.replace('pipe = TryonPipeline.from_pretrained(', patch_device + '\npipe = TryonPipeline.from_pretrained(')
src = src.replace('pipe.unet_encoder = UNet_Encoder', 'pipe.unet_encoder = UNet_Encoder\ntry:\n    pipe.enable_vae_tiling()\nexcept:\n    pass')

old_gpu_init = "    openpose_model.preprocessor.body_estimation.model.to(device)\n    pipe.to(device)\n    pipe.unet_encoder.to(device)"
new_gpu_init = """    import gc
    pipe.unet.to('cpu')
    pipe.vae.to('cpu')
    pipe.image_encoder.to('cpu')
    pipe.text_encoder.to('cpu')
    pipe.text_encoder_2.to('cpu')
    pipe.unet_encoder.to('cpu')
    openpose_model.preprocessor.body_estimation.model.to('cpu')
    gc.collect()
    torch.cuda.empty_cache()"""
src = src.replace(old_gpu_init, new_gpu_init)

old_is_checked = "    if is_checked:\n        keypoints = openpose_model(human_img.resize((384,512)))\n        model_parse, _ = parsing_model(human_img.resize((384,512)))"
new_is_checked = """    if is_checked:
        openpose_model.preprocessor.body_estimation.model.to(device)
        keypoints = openpose_model(human_img.resize((384,512)))
        openpose_model.preprocessor.body_estimation.model.to('cpu')
        model_parse, _ = parsing_model(human_img.resize((384,512)))
        gc.collect()
        torch.cuda.empty_cache()"""
src = src.replace(old_is_checked, new_is_checked)

old_encode = "    with torch.no_grad():\n        # Extract the images\n        with torch.cuda.amp.autocast():\n            with torch.no_grad():\n                prompt = 'model is wearing ' + garment_des"
new_encode = """    with torch.no_grad():
        # Extract the images
        with torch.cuda.amp.autocast():
            with torch.no_grad():
                pipe.text_encoder.to(device)
                pipe.text_encoder_2.to(device)
                prompt = 'model is wearing ' + garment_des"""
src = src.replace(old_encode, new_encode)

old_pipe_call = "                    pose_img =  tensor_transfrom(pose_img).unsqueeze(0).to(device,torch.float16)\n                    garm_tensor =  tensor_transfrom(garm_img).unsqueeze(0).to(device,torch.float16)\n                    generator = torch.Generator(device).manual_seed(seed) if seed is not None else None\n                    images = pipe("
new_pipe_call = """                    pipe.text_encoder.to('cpu')
                    pipe.text_encoder_2.to('cpu')
                    gc.collect()
                    torch.cuda.empty_cache()
                    pipe.unet.to(device)
                    pipe.vae.to(device)
                    pipe.image_encoder.to(device)
                    pipe.unet_encoder.to(device)
                    pose_img = tensor_transfrom(pose_img).unsqueeze(0).to(device, torch.float16)
                    garm_tensor = tensor_transfrom(garm_img).unsqueeze(0).to(device, torch.float16)
                    generator = torch.Generator(device).manual_seed(seed) if seed is not None else None
                    images = pipe("""
        src = src.replace(old_pipe_call, new_pipe_call)
        src = src.replace("base_path = 'yisol/IDM-VTON'", "base_path = '/content/IDM-VTON/ckpt/IDM-VTON-F16'")
        if 'low_cpu_mem_usage=True' not in src:
            src = src.replace('torch_dtype=torch.float16,', 'torch_dtype=torch.float16, low_cpu_mem_usage=True,')
        src = src.replace('image_blocks.launch()', 'image_blocks.launch(share=True, show_error=True)')
        if 'share=True' not in src:
            src += "\n\nimage_blocks.launch(share=True, show_error=True)\n"
        app.write_text(src)
        print('Patched IDM-VTON. Launching the public Gradio worker now.')
        !python gradio_demo/app.py
